In [1]:
!pip install -U spacy -q

In [1]:
import spacy
from spacy.tokens import DocBin
from tqdm import tqdm

nlp = spacy.blank("en")
db = DocBin()

In [2]:
import json

f = open('leave_types_annotation.json')
TRAIN_DATA = json.load(f)
print(TRAIN_DATA)

{'classes': ['SICK', 'CASUAL', 'EARNED', 'HALF DAY'], 'annotations': [['I want to apply casual leave due to a family emergency\r', {'entities': [[16, 28, 'CASUAL']]}], ['I want to apply casual leave due to a medical appointment\r', {'entities': [[16, 28, 'CASUAL']]}], ['I want to apply casual leave due to personal reasons\r', {'entities': [[16, 28, 'CASUAL']]}], ['I want to apply casual leave due to attending a wedding\r', {'entities': [[16, 28, 'CASUAL']]}], ['I want to apply casual leave due to childcare responsibilities\r', {'entities': [[16, 28, 'CASUAL']]}], ['I want to apply casual leave due to household chores\r', {'entities': [[16, 28, 'CASUAL']]}], ['I want to apply casual leave due to a mental health day\r', {'entities': [[16, 28, 'CASUAL']]}], ['I want to apply casual leave due to volunteering commitments\r', {'entities': [[16, 28, 'CASUAL']]}], ['I want to apply casual leave due to a religious observance\r', {'entities': [[16, 28, 'CASUAL']]}], ['I want to apply casual leav

In [3]:
for text, annot in tqdm(TRAIN_DATA['annotations']):
  doc =nlp.make_doc(text)
  ents= []
  for start, end, label in annot['entities']:
    span= doc.char_span(start,end,label=label,alignment_mode="contract")
    if span is None:
      print("Skpping Entity")
    else:
      ents.append(span)

  doc.ents = ents
  db.add(doc)

db.to_disk("./training_data.spacy")

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 407/407 [00:00<00:00, 6769.84it/s]


In [4]:
!python -m spacy init config config.cfg --lang en --pipeline ner --optimize efficiency

⚠ To generate a more effective transformer-based config (GPU-only),
install the spacy-transformers package and re-run this command. The config
generated now does not use transformers.
ℹ Generated config template specific for your use case
- Language: en
- Pipeline: ner
- Optimize for: efficiency
- Hardware: CPU
- Transformer: None
✔ Auto-filled config with all values
✔ Saved config
config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


In [5]:
! python -m spacy train config.cfg --output ./ --paths.train ./training_data.spacy --paths.dev ./training_data.spacy

ℹ Saving to output directory: .
ℹ Using CPU

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00     52.72    0.00    0.00    0.00    0.00
  4     200         49.22   1429.71   99.14   99.26   99.01    0.99
  9     400         21.94     46.54   99.51   99.51   99.51    1.00
 16     600         12.59     26.73  100.00  100.00  100.00    1.00
 23     800          0.44      0.35  100.00  100.00  100.00    1.00
 33    1000          2.15      1.01  100.00  100.00  100.00    1.00
 45    1200          0.00      0.00  100.00  100.00  100.00    1.00
 60    1400          0.00      0.00  100.00  100.00  100.00    1.00
 77    1600          0.00      0.00 

In [6]:
nlp_ner = spacy.load('model-best')

In [7]:
doc = nlp_ner(
    '''
i want to apply casual leave due to marriage 


    '''

)
spacy.displacy.render(doc,style="ent",jupyter=True)


In [8]:
doc = nlp_ner(
    '''
i want to apply sick leave due to having fever

    '''

)
spacy.displacy.render(doc,style="ent",jupyter=True)

In [9]:
doc = nlp_ner(
    '''
I want to have earned leave due to fanily function

    '''

)
spacy.displacy.render(doc,style="ent",jupyter=True)

In [28]:
for ent in doc.ents:
    # print(ent.text, ent.start, ent.end, ent.start_char, ent.end_char, ent.label_)
    if ent.label_ == "DIN NUMBER":
        din = ent.text
        print(din)
        break

In [11]:

doc = nlp_ner(
    '''
I want half day leave due to urgent work
    '''

)
spacy.displacy.render(doc,style="ent",jupyter=True)

In [16]:

doc = nlp_ner(
    '''
I want jacl leave due to urgent work
    '''

)
spacy.displacy.render(doc,style="ent",jupyter=True)